## Import Libraries

In [ ]:
import cv2
import numpy as np
import pandas as pd

from PIL import Image as PImage, ImageDraw as PImageDraw

from datasets import load_dataset

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

### Silhouette `Dataset`
- [visualizedata/revolutionary_silhouettes](https://huggingface.co/datasets/visualizedata/revolutionary_silhouettes)

In [ ]:
img_ds = load_dataset("visualizedata/revolutionary_silhouettes", split="sized")
img_ds

In [ ]:
sil_ds = load_dataset("visualizedata/revolutionary_silhouettes", split="silhouetted")
sil_ds

### Clustering of Silhouettes by Points

In [ ]:
xy_feats = [f for f in sil_ds.features.keys() if f.startswith(("x", "y"))]
non_xy_feats = [f for f in sil_ds.features.keys() if not f.startswith(("x", "y"))]

sil_xy_ds = sil_ds.select_columns(xy_feats)

sil_xy_df = sil_xy_ds.to_pandas()
sil_xy_df

In [ ]:
# Center points
xy_pairs = sil_xy_df.values.reshape(len(sil_xy_df), -1, 2)
xy_mean = xy_pairs.mean(axis=1).mean(axis=0)
xy_centered = (xy_pairs - xy_mean).reshape(len(sil_xy_df), -1)
sil_xy_centered_df = pd.DataFrame(xy_centered, columns=sil_xy_df.columns)

In [ ]:
n_clusters = 8
kmeans = KMeans(n_clusters=n_clusters, random_state=1010).fit(sil_xy_centered_df)

In [ ]:
r = 10

for cidx in range(n_clusters):
  aimg = PImage.fromarray(255*np.ones((2000,8000,3), dtype=np.uint8)).convert("RGBA")
  aiw,aih = aimg.size
  draw = PImageDraw.Draw(aimg)

  cluster_center = kmeans.cluster_centers_[cidx]
  cluster_sils = xy_centered[kmeans.labels_ == cidx]
  cluster_avg = cluster_sils.mean(axis=0).reshape(-1,2)
  dists = euclidean_distances([cluster_center], xy_centered)
  by_dist = dists[0].argsort()

  for x,y in cluster_center.reshape(-1,2):
    x += xy_mean[0]
    y += xy_mean[1]
    draw.ellipse((x-r, y-r, x+r, y+r), fill=(220,0,0,100))

  for cnt in range(4):
    offset = cnt * 2000 + 2000
    for x,y in xy_centered[by_dist[cnt]].reshape(-1,2):
      x += xy_mean[0]
      y += xy_mean[1]
      draw.ellipse((x-r+offset, y-r, x+r+offset, y+r), fill=(0,0,0))

  
  display(aimg.resize((800, 200)))

### Clustering of Silhouettes by Image Hu Moments

In [ ]:
#  get largest dimensions
img_dims = np.array([img.size for img in img_ds["image"]])
img_dims_max = img_dims.max(axis=0)
new_img_dims = img_dims_max // 4

# resize, center
img_resized = []
for img in img_ds["image"]:
  iw,ih = img.size
  siw,sih = iw//4, ih//4
  simg = img.resize((siw,sih))

  nx, ny = (new_img_dims[1] - siw) // 2, (new_img_dims[0] - sih) // 2

  nimg = PImage.new("RGB", (new_img_dims[1], new_img_dims[0]), (255,255,255))
  nimg.paste(simg, (nx, ny))
  img_resized.append(np.array(nimg))

In [ ]:
center_r = 10
thold_pad = 64

# threshold
img_bw = []
for img_np in img_resized:
  ih,iw,_ = img_np.shape
  center = img_np[ih//2-center_r:ih//2+center_r+1, iw//2-center_r:iw//2+center_r+1]
  center_avg = int(center.mean())
  ret, img_t_np = cv2.threshold(cv2.cvtColor(img_np, cv2.COLOR_BGR2GRAY), center_avg+thold_pad, 255, cv2.THRESH_BINARY)
  img_bw.append(img_t_np)

In [ ]:
# Hu
img_hu = []
img_hu_log = []

for img_np in img_bw:
  moments = cv2.moments(img_np)
  hu_moments = cv2.HuMoments(moments)[:-1,0]
  hu_moments_log = [np.sign(hm) * np.log10(abs(hm)) for hm in hu_moments]
  img_hu.append(hu_moments)
  img_hu_log.append(hu_moments_log)

In [ ]:
img_hu_df = pd.DataFrame(img_hu)
img_hu_log_df = pd.DataFrame(img_hu_log)

In [ ]:
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=1010).fit(img_hu_log_df)

In [ ]:
for cidx in range(n_clusters):
  cimg = PImage.fromarray(255*np.ones((247,2500,3), dtype=np.uint8)).convert("RGB")

  dists = euclidean_distances([kmeans.cluster_centers_[cidx]], img_hu_log_df)
  by_dist = dists[0].argsort(axis=0)

  cx = 0
  for img_cnt in range(10):
    img = PImage.fromarray(img_bw[by_dist[img_cnt]])
    cimg.paste(img, (cx, 0))
    cx += img.width

  cimg = cimg.crop((0,0, cx, cimg.height))
  display(cimg)